<a href="https://colab.research.google.com/github/AG-0745/UCS420/blob/main/Cognitive_Ass_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import re
import nltk
import numpy as np
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from nltk.stem import PorterStemmer, WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

#Q1

In [21]:
paragraph = (
    "Artificial intelligence is rapidly transforming the modern technological landscape. "
    "Machine learning models can now process massive datasets to uncover hidden patterns. "
    "Natural language processing bridges the communication gap between humans and machines. "
    "Developers utilize frameworks like TensorFlow and PyTorch to build intelligent systems. "
    "As computational power increases, the potential applications seem virtually limitless. "
    "Ultimately, ethical considerations must guide the future development of these powerful tools."
)

text_lower = paragraph.lower()
text_clean = re.sub(r'[^\w\s]', '', text_lower)

sentences = sent_tokenize(paragraph)
words = word_tokenize(text_clean)

split_words = text_clean.split()
print(f"Q1 - split() count: {len(split_words)} | word_tokenize() count: {len(words)}")

stop_words = set(stopwords.words('english'))
filtered_words = [w for w in words if w not in stop_words]

fdist = FreqDist(filtered_words)
print("Q1 Top 3 Freq:", fdist.most_common(3))

Q1 - split() count: 65 | word_tokenize() count: 65
Q1 Top 3 Freq: [('artificial', 1), ('intelligence', 1), ('rapidly', 1)]


#Q2

In [22]:
words_alpha = re.findall(r'\b[A-Za-z]+\b', paragraph.lower())
filtered_q2 = [w for w in words_alpha if w not in stop_words]

stemmer = PorterStemmer()
stemmed = [stemmer.stem(w) for w in filtered_q2]

lemmatizer = WordNetLemmatizer()
lemmatized = [lemmatizer.lemmatize(w) for w in filtered_q2]

print("\nQ2 Sample (Orig -> Stem -> Lemma):")
for o, s, l in list(zip(filtered_q2, stemmed, lemmatized))[:3]:
    print(f"  {o} -> {s} -> {l}")


Q2 Sample (Orig -> Stem -> Lemma):
  artificial -> artifici -> artificial
  intelligence -> intellig -> intelligence
  rapidly -> rapidli -> rapidly


#Q3

In [23]:
texts_q3 = [
    "Artificial intelligence and machine learning are revolutionizing modern technology.",
    "Python is widely used for data science and natural language processing tasks.",
    "Cloud computing and deep learning frameworks enable scalable AI systems."
]

cv = CountVectorizer()
bow = cv.fit_transform(texts_q3)

tv = TfidfVectorizer()
tfidf = tv.fit_transform(texts_q3)
feat_names = tv.get_feature_names_out()

print("\nQ3 Top Keywords per Text:")
for i in range(len(texts_q3)):
    top_idx = np.argsort(tfidf[i].toarray()[0])[::-1][:3]
    keywords = [feat_names[idx] for idx in top_idx]
    print(f"  Text {i+1}: {keywords}")


Q3 Top Keywords per Text:
  Text 1: ['technology', 'revolutionizing', 'modern']
  Text 2: ['widely', 'used', 'tasks']
  Text 3: ['scalable', 'systems', 'frameworks']


#Q4

In [24]:
text1 = "Artificial intelligence enables machines to mimic human intelligence and cognitive functions through advanced machine learning algorithms."
text2 = "Blockchain is a decentralized and distributed ledger technology that securely records transactions using cryptography."

def get_tokens(t):
    return [w for w in word_tokenize(re.sub(r'[^\w\s]', '', t.lower())) if w not in stop_words]

set1, set2 = set(get_tokens(text1)), set(get_tokens(text2))
jaccard = len(set1.intersection(set2)) / len(set1.union(set2))

tfidf_q4 = tv.fit_transform([text1, text2])
cosine = cosine_similarity(tfidf_q4[0:1], tfidf_q4[1:2])[0][0]

print(f"\nQ4 - Jaccard: {jaccard:.4f} | Cosine: {cosine:.4f}")


Q4 - Jaccard: 0.0000 | Cosine: 0.0342


#Q5

In [25]:
reviews = [
    "An absolute masterpiece! The battery life is phenomenal.",
    "Terrible customer service and the product stopped working."
]

print("\nQ5 Sentiments:")
for r in reviews:
    pol = TextBlob(r).sentiment.polarity
    sentiment = "Positive" if pol > 0 else "Negative"
    print(f"  [{sentiment} (Polarity: {pol:.2f})] {r}")


Q5 Sentiments:
  [Positive (Polarity: 0.38)] An absolute masterpiece! The battery life is phenomenal.
  [Negative (Polarity: -1.00)] Terrible customer service and the product stopped working.


#Q6

In [26]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([paragraph])
total_words = len(tokenizer.word_index) + 1

seqs = []
toks = tokenizer.texts_to_sequences([paragraph])[0]
for i in range(1, len(toks)):
    seqs.append(toks[:i+1])

max_len = max([len(x) for x in seqs])
seqs = pad_sequences(seqs, maxlen=max_len, padding='pre')

X, y = seqs[:, :-1], tf.keras.utils.to_categorical(seqs[:, -1], num_classes=total_words)

model = Sequential([
    Embedding(total_words, 10, input_length=max_len-1),
    LSTM(30),
    Dense(total_words, activation='softmax')
])
model.compile(loss='categorical_crossentropy', optimizer='adam')
model.fit(X, y, epochs=5, verbose=0)

seed = "artificial intelligence"
for _ in range(5):
    encoded = pad_sequences([tokenizer.texts_to_sequences([seed])[0]], maxlen=max_len-1, padding='pre')
    pred = tf.argmax(model.predict(encoded, verbose=0), axis=-1).numpy()[0]
    for w, idx in tokenizer.word_index.items():
        if idx == pred:
            seed += " " + w
            break

print(f"\nQ6 Generated Text: {seed}")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(



Q6 Generated Text: artificial intelligence the the the the the
